# Lab 04-02: Vector Search Index

**Module:** 04 -- Assembling & Deploying Applications (22% of exam)  
**UI Sections:** Catalog | Compute | Vector Search  
**Estimated Time:** 50--60 minutes  
**Difficulty:** Intermediate

---

## What and Why

Vector Search indexes are the retrieval engine in RAG. When a user asks a question, the
retriever converts that question into a vector (embedding), searches the index for the
most similar document chunks, and returns them as context for the LLM.

The exam **heavily** tests the two index types (Delta Sync vs Direct Vector Access), when
to use each, and the role of Change Data Feed. If you understand these concepts cold, you
will answer 4--6 questions correctly that others miss.

---

## Mental Model

> **Analogy:** A Vector Search index is like a library card catalog that understands
> meaning. Instead of searching by exact title or author, you describe what you are
> looking for ("something about quantum computing for beginners") and it finds the
> closest matches. Delta Sync = auto-updating catalog that stays in sync with new
> book arrivals; Direct Vector Access = you manually update the cards yourself.

---

## Exam Alert

| Trap (Wrong) | Correct Answer |
|---|---|
| "Delta Sync index requires manual updates" | Delta Sync auto-syncs via Change Data Feed -- that is the whole point |
| "Direct Vector Access is always better because you control it" | Delta Sync is recommended for most RAG use cases (simpler, always current) |
| "Any Delta table can be a Vector Search source" | The table must have CDF enabled (`delta.enableChangeDataFeed = true`) |
| "Vector Search creates embeddings for you automatically" | For Delta Sync you specify an embedding model endpoint; for Direct Access you provide pre-computed vectors |
| "Vector Search and Model Serving share the same endpoint" | Vector Search endpoints are separate compute resources from Model Serving endpoints |

---

## Learning Objectives

By the end of this lab you will be able to:

1. Create a Vector Search endpoint (compute resource)
2. Create a Delta Sync Index with managed embeddings
3. Create a Direct Vector Access Index with pre-computed vectors
4. Query an index using `similarity_search()`
5. Compare the two index types and choose the right one for a use case
6. Manage indexes: check sync status, rebuild, delete

---

## Exam Objectives Covered

| Objective | Tested Here |
|---|---|
| Create Vector Search endpoints | Step 1 |
| Create Delta Sync indexes | Step 2 |
| Create Direct Vector Access indexes | Step 3 |
| Query vector indexes | Steps 4--5 |
| Compare index types | Step 6 |
| Manage index lifecycle | Step 7 |

---

## Step 1: Prerequisites -- Your Delta Table with CDF

Before creating a Vector Search index, you need a Delta table with Change Data Feed
enabled. If you completed Lab 02-03, you already have `workspace.genai_labs.document_chunks`.
If not, here is how to create one from scratch.

### Why CDF is Required

Delta Sync indexes read the Change Data Feed to know which rows were added, updated, or
deleted since the last sync. Without CDF, the index has no way to track changes and will
refuse to sync.

```sql
-- Enable CDF on an existing table
ALTER TABLE workspace.genai_labs.document_chunks
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
```

In [ ]:
# ---------------------------------------------------------------------------
# Create a Delta table with CDF enabled (if not done in Lab 02-03)
# ---------------------------------------------------------------------------
# On Databricks:
#
# spark.sql("""
#     CREATE TABLE IF NOT EXISTS workspace.genai_labs.document_chunks (
#         chunk_id    STRING NOT NULL,
#         source_doc  STRING NOT NULL,
#         chunk_text  STRING NOT NULL,
#         chunk_index INT    NOT NULL,
#         token_count INT    NOT NULL
#     )
#     USING DELTA
#     TBLPROPERTIES (delta.enableChangeDataFeed = true)
# """)
#
# # Verify CDF is enabled
# spark.sql("SHOW TBLPROPERTIES workspace.genai_labs.document_chunks").show(truncate=False)

# Local simulation -- show the table we'll work with
import pandas as pd
import uuid

TABLE_NAME = "workspace.genai_labs.document_chunks"

sample_chunks = pd.DataFrame({
    "chunk_id": [str(uuid.uuid4()) for _ in range(6)],
    "source_doc": [
        "databricks_overview.txt", "databricks_overview.txt",
        "vector_search_guide.txt", "vector_search_guide.txt",
        "model_serving_basics.txt", "model_serving_basics.txt",
    ],
    "chunk_text": [
        "Databricks is a unified analytics platform built on Apache Spark.",
        "Unity Catalog provides fine-grained access control and data lineage.",
        "Vector Search lets you create search indexes on Delta tables.",
        "Delta Sync Index automatically syncs with the source Delta table.",
        "Model Serving deploys ML models as REST API endpoints.",
        "Serving endpoints support auto-scaling and traffic splitting.",
    ],
    "chunk_index": [0, 1, 0, 1, 0, 1],
    "token_count": [14, 13, 12, 11, 11, 10],
})

print(f"Source table: {TABLE_NAME}")
print(f"Rows: {len(sample_chunks)}")
print(f"CDF enabled: True (required for Delta Sync)")
print()
print(sample_chunks[["source_doc", "chunk_index", "chunk_text"]].to_string(index=False))


**Expected output:**

```
Source table: workspace.genai_labs.document_chunks
Rows: 6
CDF enabled: True (required for Delta Sync)

             source_doc  chunk_index                                                       chunk_text
databricks_overview.txt            0  Databricks is a unified analytics platform built on Apache Spark.
databricks_overview.txt            1  Unity Catalog provides fine-grained access control and data lineage.
vector_search_guide.txt            0  Vector Search lets you create search indexes on Delta tables.
...
```

---

## Step 2: Creating a Vector Search Endpoint

A Vector Search **endpoint** is a compute resource that hosts your indexes. Think of it
as the server that runs your searches. It is separate from:
- **Model Serving endpoints** (which serve ML models)
- **SQL Warehouses** (which run SQL queries)
- **All-purpose clusters** (which run notebooks)

### Why a Separate Endpoint?

Vector search is a specialized workload (approximate nearest neighbor search) that
benefits from dedicated compute. The endpoint scales independently of your other
workloads.

### UI Navigation

**UI -->** Left nav --> **Compute** --> **Vector Search** tab --> **Create**

You will see:
- **Endpoint name**: a unique identifier (e.g., `genai_vs_endpoint`)
- **Endpoint type**: determines scaling behavior

In [ ]:
# ---------------------------------------------------------------------------
# Create a Vector Search endpoint
# ---------------------------------------------------------------------------
# On Databricks:
#
# from databricks.vector_search.client import VectorSearchClient
#
# vsc = VectorSearchClient()
#
# # Create the endpoint
# vsc.create_endpoint(
#     name="genai_vs_endpoint",
#     endpoint_type="STANDARD",   # STANDARD is the default
# )
#
# # Wait for it to be ready
# vsc.wait_until_endpoint_is_ready("genai_vs_endpoint")
#
# # List all endpoints
# endpoints = vsc.list_endpoints()
# for ep in endpoints.get("endpoints", []):
#     print(f"  {ep['name']} - {ep['endpoint_status']['state']}")

VS_ENDPOINT = "genai_vs_endpoint"

print("Vector Search Endpoint Creation")
print("=" * 50)
print()
print("from databricks.vector_search.client import VectorSearchClient")
print()
print("vsc = VectorSearchClient()")
print()
print(f'vsc.create_endpoint(')
print(f'    name="{VS_ENDPOINT}",')
print(f'    endpoint_type="STANDARD",')
print(f')')
print()
print(f"Endpoint: {VS_ENDPOINT}")
print(f"Status: PROVISIONING --> ONLINE (takes 5-10 minutes)")
print()
print("UI Navigation:")
print("  **UI -->** Left nav --> Compute --> Vector Search tab")
print(f"  You should see '{VS_ENDPOINT}' with status ONLINE.")


**Expected output:**

```
Vector Search Endpoint Creation
==================================================

from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

vsc.create_endpoint(
    name="genai_vs_endpoint",
    endpoint_type="STANDARD",
)

Endpoint: genai_vs_endpoint
Status: PROVISIONING --> ONLINE (takes 5-10 minutes)
```

### Key Concept: Endpoint vs. Index

| Concept | What It Is | Analogy |
|---|---|---|
| **Endpoint** | Compute resource that hosts indexes | The server room |
| **Index** | The actual searchable data structure | A specific filing cabinet in the room |

One endpoint can host multiple indexes. You create the endpoint first, then create
indexes on it.

---

## Step 3: Delta Sync Index -- Automatic Synchronization

A **Delta Sync Index** automatically stays in sync with its source Delta table via
Change Data Feed. When you INSERT, UPDATE, or DELETE rows in the table, the index
detects the changes and updates accordingly.

### Two Embedding Modes

| Mode | Embeddings Computed By | Source Column | When to Use |
|---|---|---|---|
| **Managed embeddings** | Databricks (via an embedding model endpoint) | Text column (e.g., `chunk_text`) | Most RAG use cases -- simplest approach |
| **Self-managed embeddings** | You (pre-computed and stored in the table) | Vector column (e.g., `embedding`) | When you need a specific embedding model not on Databricks |

### Managed Embeddings: How It Works

1. You specify a text column (e.g., `chunk_text`) and an embedding model endpoint
   (e.g., `databricks-bge-large-en`)
2. When the index syncs, it sends each chunk to the embedding model endpoint
3. The resulting vectors are stored in the index
4. At query time, the query text is also embedded using the same model

> **Analogy:** Managed embeddings are like a library that catalogs books for you.
> You just drop off the book (text), and the librarian (embedding model) creates
> the card catalog entry (vector). Self-managed is like writing your own catalog
> cards and handing them to the librarian.

In [ ]:
# ---------------------------------------------------------------------------
# Create a Delta Sync Index with managed embeddings
# ---------------------------------------------------------------------------
# On Databricks:
#
# index = vsc.create_delta_sync_index(
#     endpoint_name="genai_vs_endpoint",
#     index_name="workspace.genai_labs.document_chunks_index",
#     source_table_name="workspace.genai_labs.document_chunks",
#     pipeline_type="TRIGGERED",       # TRIGGERED or CONTINUOUS
#     primary_key="chunk_id",
#     embedding_source_column="chunk_text",
#     embedding_model_endpoint_name="databricks-bge-large-en",
# )
#
# # Wait for initial sync
# import time
# while True:
#     status = index.describe()
#     state = status.get("status", {}).get("detailed_state")
#     print(f"  Sync status: {state}")
#     if state == "ONLINE":
#         break
#     time.sleep(30)

INDEX_NAME = "workspace.genai_labs.document_chunks_index"
EMBEDDING_MODEL = "databricks-bge-large-en"

print("Delta Sync Index Creation (Managed Embeddings)")
print("=" * 55)
print()
print(f'index = vsc.create_delta_sync_index(')
print(f'    endpoint_name="{VS_ENDPOINT}",')
print(f'    index_name="{INDEX_NAME}",')
print(f'    source_table_name="{TABLE_NAME}",')
print(f'    pipeline_type="TRIGGERED",')
print(f'    primary_key="chunk_id",')
print(f'    embedding_source_column="chunk_text",')
print(f'    embedding_model_endpoint_name="{EMBEDDING_MODEL}",')
print(f')')
print()
print("Key parameters:")
print(f"  endpoint_name: {VS_ENDPOINT} (the compute hosting this index)")
print(f"  source_table_name: {TABLE_NAME} (must have CDF enabled)")
print(f"  pipeline_type: TRIGGERED (sync on demand) or CONTINUOUS (auto-sync)")
print(f"  primary_key: chunk_id (unique identifier for each row)")
print(f"  embedding_source_column: chunk_text (text to embed)")
print(f"  embedding_model_endpoint_name: {EMBEDDING_MODEL} (computes vectors)")


**Expected output:**

```
Delta Sync Index Creation (Managed Embeddings)
=======================================================

index = vsc.create_delta_sync_index(
    endpoint_name="genai_vs_endpoint",
    index_name="workspace.genai_labs.document_chunks_index",
    source_table_name="workspace.genai_labs.document_chunks",
    pipeline_type="TRIGGERED",
    primary_key="chunk_id",
    embedding_source_column="chunk_text",
    embedding_model_endpoint_name="databricks-bge-large-en",
)
```

### Pipeline Type: TRIGGERED vs. CONTINUOUS

| Pipeline Type | Behavior | Cost | Use Case |
|---|---|---|---|
| **TRIGGERED** | Syncs only when you manually trigger or schedule | Lower (runs only when triggered) | Batch updates, periodic ingestion |
| **CONTINUOUS** | Syncs automatically within seconds of table changes | Higher (always-on pipeline) | Real-time ingestion, live data |

In [ ]:
# ---------------------------------------------------------------------------
# Delta Sync Index with self-managed embeddings
# (for when you pre-compute vectors yourself)
# ---------------------------------------------------------------------------
# If your Delta table already has an embedding column:
#
# CREATE TABLE workspace.genai_labs.chunks_with_vectors (
#     chunk_id    STRING NOT NULL,
#     chunk_text  STRING NOT NULL,
#     embedding   ARRAY<FLOAT> NOT NULL,   -- pre-computed vector
#     source_doc  STRING
# )
# USING DELTA
# TBLPROPERTIES (delta.enableChangeDataFeed = true);
#
# index = vsc.create_delta_sync_index(
#     endpoint_name="genai_vs_endpoint",
#     index_name="workspace.genai_labs.chunks_with_vectors_index",
#     source_table_name="workspace.genai_labs.chunks_with_vectors",
#     pipeline_type="TRIGGERED",
#     primary_key="chunk_id",
#     embedding_dimension=1024,       # Must match your embedding model's output
#     embedding_vector_column="embedding",   # Column with pre-computed vectors
# )

print("Delta Sync Index with SELF-MANAGED Embeddings")
print("=" * 50)
print()
print("Key difference from managed embeddings:")
print("  - No embedding_model_endpoint_name")
print("  - Instead, specify embedding_vector_column and embedding_dimension")
print("  - You are responsible for computing and storing the vectors")
print()
print("Parameters:")
print('  embedding_vector_column="embedding"  # column with ARRAY<FLOAT>')
print('  embedding_dimension=1024              # must match your model output')
print()
print("When to use self-managed:")
print("  - You use a custom embedding model not available on Databricks")
print("  - You need to preprocess vectors before indexing")
print("  - You want exact control over the embedding pipeline")


**Expected output:**

```
Delta Sync Index with SELF-MANAGED Embeddings
==================================================

Key difference from managed embeddings:
  - No embedding_model_endpoint_name
  - Instead, specify embedding_vector_column and embedding_dimension
  - You are responsible for computing and storing the vectors
```

### Managed vs. Self-Managed Embeddings (Delta Sync)

| Feature | Managed | Self-Managed |
|---|---|---|
| You provide | Text column | Vector column (ARRAY<FLOAT>) |
| Embedding computed by | Databricks (embedding model endpoint) | You (before writing to table) |
| Key parameter | `embedding_model_endpoint_name` | `embedding_vector_column` + `embedding_dimension` |
| Simplicity | Simplest -- just point at text | More work -- you manage the pipeline |
| Flexibility | Limited to Databricks-hosted models | Any embedding model |

---

## Step 4: Direct Vector Access Index -- Manual Control

A **Direct Vector Access Index** gives you complete control. There is no source Delta
table -- instead, you push vectors directly to the index via API calls (`upsert` and
`delete`). The index is a standalone data structure.

### When to Use Direct Vector Access

- Your vectors come from an external system (not a Delta table)
- You need real-time updates without waiting for CDF sync
- You are building a custom pipeline that does not use Delta Lake

### Key Differences from Delta Sync

| Feature | Delta Sync | Direct Vector Access |
|---|---|---|
| Source | Delta table | None (you push data via API) |
| Sync | Automatic (CDF) | Manual (upsert/delete API calls) |
| Embeddings | Managed or self-managed | Self-managed only |
| CDF required | Yes | No (no Delta table involved) |
| Recommended for | Most RAG use cases | Custom/external pipelines |

In [ ]:
# ---------------------------------------------------------------------------
# Create a Direct Vector Access Index
# ---------------------------------------------------------------------------
# On Databricks:
#
# direct_index = vsc.create_direct_access_index(
#     endpoint_name="genai_vs_endpoint",
#     index_name="workspace.genai_labs.custom_vectors_index",
#     primary_key="chunk_id",
#     embedding_dimension=1024,
#     embedding_vector_column="embedding",
#     schema={
#         "chunk_id": "string",
#         "chunk_text": "string",
#         "source_doc": "string",
#         "embedding": "array<float>",
#     },
# )

DIRECT_INDEX_NAME = "workspace.genai_labs.custom_vectors_index"

print("Direct Vector Access Index Creation")
print("=" * 50)
print()
print(f'direct_index = vsc.create_direct_access_index(')
print(f'    endpoint_name="{VS_ENDPOINT}",')
print(f'    index_name="{DIRECT_INDEX_NAME}",')
print(f'    primary_key="chunk_id",')
print(f'    embedding_dimension=1024,')
print(f'    embedding_vector_column="embedding",')
print(f'    schema={{')
print(f'        "chunk_id": "string",')
print(f'        "chunk_text": "string",')
print(f'        "source_doc": "string",')
print(f'        "embedding": "array<float>",')
print(f'    }},')
print(f')')
print()
print("Key differences from Delta Sync:")
print("  - No source_table_name (no Delta table)")
print("  - No pipeline_type (no sync pipeline)")
print("  - You must define the schema explicitly")
print("  - Embeddings are always self-managed")


**Expected output:**

```
Direct Vector Access Index Creation
==================================================

direct_index = vsc.create_direct_access_index(
    endpoint_name="genai_vs_endpoint",
    index_name="workspace.genai_labs.custom_vectors_index",
    primary_key="chunk_id",
    embedding_dimension=1024,
    embedding_vector_column="embedding",
    schema={
        "chunk_id": "string",
        ...
    },
)
```

In [ ]:
# ---------------------------------------------------------------------------
# Upsert data into a Direct Vector Access Index
# ---------------------------------------------------------------------------
# On Databricks:
#
# import numpy as np
#
# # Generate sample embeddings (in production, use your embedding model)
# sample_data = [
#     {
#         "chunk_id": "chunk_001",
#         "chunk_text": "Databricks is a unified analytics platform.",
#         "source_doc": "overview.txt",
#         "embedding": np.random.randn(1024).tolist(),
#     },
#     {
#         "chunk_id": "chunk_002",
#         "chunk_text": "Unity Catalog provides data governance.",
#         "source_doc": "overview.txt",
#         "embedding": np.random.randn(1024).tolist(),
#     },
# ]
#
# # Upsert: insert new rows or update existing ones (matched by primary_key)
# direct_index.upsert(sample_data)
#
# # Delete specific rows
# direct_index.delete(["chunk_001"])

print("Upserting Data into Direct Vector Access Index")
print("=" * 50)
print()
print("# Upsert (insert or update) rows:")
print('direct_index.upsert([')
print('    {"chunk_id": "chunk_001", "chunk_text": "...", "embedding": [0.1, 0.2, ...]},') 
print('    {"chunk_id": "chunk_002", "chunk_text": "...", "embedding": [0.3, 0.4, ...]},') 
print('])')
print()
print("# Delete rows by primary key:")
print('direct_index.delete(["chunk_001"])')
print()
print("Key points:")
print("  - upsert() = INSERT if new, UPDATE if primary_key exists")
print("  - delete() = remove rows by primary key")
print("  - You must include the embedding vector in every upsert")
print("  - Changes are immediate (no sync delay)")


**Expected output:**

```
Upserting Data into Direct Vector Access Index
==================================================

# Upsert (insert or update) rows:
direct_index.upsert([...])

# Delete rows by primary key:
direct_index.delete(["chunk_001"])
```

---

## Step 5: Querying the Index -- similarity_search()

Once your index is created and populated, you query it using `similarity_search()`. This
is the core retrieval operation in RAG: given a user question, find the most similar
document chunks.

### How similarity_search Works

1. Your query text is converted to a vector (by the embedding model)
2. The index performs Approximate Nearest Neighbor (ANN) search
3. The top-N most similar vectors are returned, along with their metadata

### Key Parameters

| Parameter | Purpose | Example |
|---|---|---|
| `query_text` | The text to search for (auto-embedded for managed indexes) | `"What is Unity Catalog?"` |
| `query_vector` | Pre-computed query vector (for self-managed indexes) | `[0.1, 0.2, ...]` |
| `columns` | Which metadata columns to return | `["chunk_text", "source_doc"]` |
| `num_results` | How many results to return | `3` |
| `filters` | Optional filters on metadata columns | `{"source_doc": "overview.txt"}` |

In [ ]:
# ---------------------------------------------------------------------------
# Query a Delta Sync Index (managed embeddings)
# ---------------------------------------------------------------------------
# On Databricks:
#
# # Get a reference to the index
# index = vsc.get_index(
#     endpoint_name="genai_vs_endpoint",
#     index_name="workspace.genai_labs.document_chunks_index",
# )
#
# # Query with text (auto-embedded by the managed embedding model)
# results = index.similarity_search(
#     query_text="What is Unity Catalog?",
#     columns=["chunk_text", "source_doc", "chunk_index"],
#     num_results=3,
# )
#
# # The result is a dict with a nested structure
# for row in results["result"]["data_array"]:
#     print(f"  Score: {row[-1]:.4f}")
#     print(f"  Text:  {row[0][:80]}...")
#     print(f"  Source: {row[1]}")
#     print()

print("Querying a Delta Sync Index (managed embeddings)")
print("=" * 55)
print()
print('results = index.similarity_search(')
print('    query_text="What is Unity Catalog?",')
print('    columns=["chunk_text", "source_doc", "chunk_index"],')
print('    num_results=3,')
print(')')
print()
print("Simulated results:")
print("-" * 55)

# Simulated results
sim_results = [
    ("Unity Catalog provides fine-grained access control and data lineage.", "databricks_overview.txt", 1, 0.92),
    ("Databricks is a unified analytics platform built on Apache Spark.", "databricks_overview.txt", 0, 0.78),
    ("Delta Lake provides ACID transactions and schema enforcement.", "databricks_overview.txt", 2, 0.71),
]

for text, source, idx, score in sim_results:
    print(f"  Score: {score:.4f}")
    print(f"  Text:  {text}")
    print(f"  Source: {source} (chunk {idx})")
    print()


**Expected output:**

```
Querying a Delta Sync Index (managed embeddings)
=======================================================

results = index.similarity_search(
    query_text="What is Unity Catalog?",
    columns=["chunk_text", "source_doc", "chunk_index"],
    num_results=3,
)

Simulated results:
-------------------------------------------------------
  Score: 0.9200
  Text:  Unity Catalog provides fine-grained access control and data lineage.
  Source: databricks_overview.txt (chunk 1)

  Score: 0.7800
  Text:  Databricks is a unified analytics platform built on Apache Spark.
  Source: databricks_overview.txt (chunk 0)
  ...
```

### Result Structure

The `similarity_search()` response is a nested dictionary:

```python
{
    "result": {
        "row_count": 3,
        "data_array": [
            ["chunk_text_value", "source_doc_value", chunk_index_value, similarity_score],
            ...
        ]
    },
    "manifest": { ... },
    "status": { ... }
}
```

The last element in each row of `data_array` is the similarity score (higher = more similar).

In [ ]:
# ---------------------------------------------------------------------------
# Query with filters -- narrow results by metadata
# ---------------------------------------------------------------------------
# On Databricks:
#
# # Filter to only search within a specific document
# filtered_results = index.similarity_search(
#     query_text="How does auto-scaling work?",
#     columns=["chunk_text", "source_doc"],
#     num_results=3,
#     filters={"source_doc": "model_serving_basics.txt"},  # metadata filter
# )

print("Querying with Metadata Filters")
print("=" * 50)
print()
print('filtered_results = index.similarity_search(')
print('    query_text="How does auto-scaling work?",')
print('    columns=["chunk_text", "source_doc"],')
print('    num_results=3,')
print('    filters={"source_doc": "model_serving_basics.txt"},')
print(')')
print()
print("Simulated filtered results (only model_serving_basics.txt):")
print("-" * 55)

filtered_sim = [
    ("Serving endpoints support auto-scaling and traffic splitting.", "model_serving_basics.txt", 0.89),
    ("Model Serving deploys ML models as REST API endpoints.", "model_serving_basics.txt", 0.72),
]

for text, source, score in filtered_sim:
    print(f"  Score: {score:.4f}  |  {text}")
print()
print("Filters are useful when:")
print("  - You want to search within a specific document or category")
print("  - You need to restrict results by tenant, date, or permission level")
print("  - You want to implement access-controlled retrieval")


**Expected output:**

```
Querying with Metadata Filters
==================================================

Simulated filtered results (only model_serving_basics.txt):
-------------------------------------------------------
  Score: 0.8900  |  Serving endpoints support auto-scaling and traffic splitting.
  Score: 0.7200  |  Model Serving deploys ML models as REST API endpoints.
```

In [ ]:
# ---------------------------------------------------------------------------
# Query a Direct Vector Access Index (self-managed embeddings)
# ---------------------------------------------------------------------------
# For self-managed embeddings, you provide the query vector, not query text.
#
# On Databricks:
#
# import numpy as np
#
# # Step 1: Embed the query yourself
# query_embedding = your_embedding_model.encode("What is Unity Catalog?")
#
# # Step 2: Search with the pre-computed vector
# results = direct_index.similarity_search(
#     query_vector=query_embedding.tolist(),
#     columns=["chunk_text", "source_doc"],
#     num_results=3,
# )

print("Querying a Direct Vector Access Index")
print("=" * 50)
print()
print("Key difference: use query_vector instead of query_text")
print()
print('# Step 1: Embed the query with your own model')
print('query_embedding = your_model.encode("What is Unity Catalog?")')
print()
print('# Step 2: Search with the vector')
print('results = direct_index.similarity_search(')
print('    query_vector=query_embedding.tolist(),')
print('    columns=["chunk_text", "source_doc"],')
print('    num_results=3,')
print(')')
print()
print("When to use query_text vs query_vector:")
print("  query_text   --> Managed embeddings (index embeds for you)")
print("  query_vector --> Self-managed embeddings (you embed yourself)")


**Expected output:**

```
Querying a Direct Vector Access Index
==================================================

Key difference: use query_vector instead of query_text

# Step 1: Embed the query with your own model
query_embedding = your_model.encode("What is Unity Catalog?")

# Step 2: Search with the vector
results = direct_index.similarity_search(
    query_vector=query_embedding.tolist(),
    ...
)
```

### Step 5b: The `.query()` API -- Exam Question Pattern

> **Exam Q55:** *An engineer needs to implement semantic search on a Databricks
> Vector Search index. What command should be used?*
>
> - (A) `vector_index.get_nearest_neighbors(query_embedding, limit=5)` -- **WRONG**: not a real method
> - (B) `vector_index.semantic_match(text=query, k=5)` -- **WRONG**: not a real method
> - **(C) `vector_index.query(embedding=query_embedding, top_k=5)`** -- **CORRECT**
> - (D) `vector_index.search("keyword-based input", top_k=5)` -- **WRONG**: keyword search, not semantic

The Databricks Vector Search SDK provides **two query methods**:

| Method | When to Use | Key Parameter |
|---|---|---|
| `index.similarity_search(query_text=...)` | **Managed embeddings** -- the index embeds your text automatically | `query_text` (string) |
| `index.similarity_search(query_vector=...)` | **Self-managed embeddings** -- you provide a pre-computed vector | `query_vector` (list of floats) |

The exam uses `.query(embedding=..., top_k=...)` as shorthand for the
vector-based search pattern. The key concept: **semantic search requires
a pre-computed embedding vector**, not raw text or keywords.

In [ ]:
# Step 5b: Vector Search Query API -- The Exam Pattern
#
# This cell shows the EXACT code patterns tested on the exam
# and explains why each distractor is wrong.

print("=" * 70)
print("VECTOR SEARCH QUERY API -- Exam Command Reference")
print("=" * 70)

print("""
  ┌─────────────────────────────────────────────────────────────────┐
  │  CORRECT: Semantic search with a pre-computed embedding vector  │
  └─────────────────────────────────────────────────────────────────┘

  # Pattern 1: Self-managed embeddings (you embed the query yourself)
  results = index.similarity_search(
      query_vector=query_embedding,   # <-- pre-computed vector
      columns=["chunk_text", "source"],
      num_results=5,
  )

  # Pattern 2: Managed embeddings (the index embeds for you)
  results = index.similarity_search(
      query_text="What is Unity Catalog?",  # <-- raw text, auto-embedded
      columns=["chunk_text", "source"],
      num_results=5,
  )

  ┌─────────────────────────────────────────────────────────────────┐
  │  WRONG answers on the exam (and why)                           │
  └─────────────────────────────────────────────────────────────────┘

  get_nearest_neighbors()  -- Not a Databricks API method.
  semantic_match()         -- Not a Databricks API method.
  .search("keyword...")   -- This would be keyword/BM25 search,
                             NOT semantic (vector-based) search.

  The KEY distinction:
    - Semantic search: uses EMBEDDING VECTORS (cosine similarity)
    - Keyword search:  uses TEXT MATCHING (BM25, exact words)
    - The exam asks about SEMANTIC search -> needs an embedding vector
""")

# Simulate the query flow
print("=" * 70)
print("QUERY FLOW: How semantic search works end-to-end")
print("=" * 70)
print("""
  User question: "How do I reset my password?"
         |
         v
  Step 1: EMBED the query  (via embedding model endpoint)
         query_vec = embedding_model.encode("How do I reset my password?")
         -> [0.021, -0.045, 0.089, ...]  (1024-dim vector)
         |
         v
  Step 2: SEARCH the index  (via .query() or .similarity_search())
         results = index.similarity_search(
             query_vector=query_vec,
             num_results=5
         )
         |
         v
  Step 3: RETURN top-K nearest neighbors (ranked by cosine similarity)
         ["Go to Settings > Reset Password...",  score: 0.92]
         ["Account security FAQ...",              score: 0.78]
""")


---

## Step 6: Comparing the Two Index Types -- Decision Table

This is one of the most heavily tested topics on the exam. You must be able to choose
the right index type for a given scenario.

### The Complete Comparison

| Feature | Delta Sync Index | Direct Vector Access Index |
|---|---|---|
| **Source** | Delta table (Unity Catalog) | None -- you push data via API |
| **Sync mechanism** | Automatic via Change Data Feed | Manual (upsert/delete API) |
| **CDF required?** | Yes | No |
| **Embedding modes** | Managed OR self-managed | Self-managed only |
| **Pipeline types** | TRIGGERED or CONTINUOUS | N/A |
| **Update latency** | Seconds (CONTINUOUS) to manual (TRIGGERED) | Immediate (API call) |
| **Schema definition** | Inferred from source table | Explicitly defined at creation |
| **Best for** | Standard RAG with Delta tables | Custom pipelines, external data |
| **Recommended?** | Yes -- for most use cases | Only when Delta Sync is not feasible |

In [ ]:
# ---------------------------------------------------------------------------
# Exam scenario practice: which index type?
# ---------------------------------------------------------------------------
scenarios = [
    {
        "scenario": "Your company stores document chunks in a Delta table and wants\n"
                    "the search index to automatically update when new documents are added.",
        "answer": "Delta Sync Index (CONTINUOUS pipeline)",
        "why": "Auto-sync via CDF is the defining feature of Delta Sync.",
    },
    {
        "scenario": "You receive pre-computed embeddings from an external ML service\n"
                    "and need to push them into a search index in real-time.",
        "answer": "Direct Vector Access Index",
        "why": "No Delta table source; vectors come from external system via API.",
    },
    {
        "scenario": "You have a Delta table with text chunks and want Databricks to\n"
                    "handle embedding computation using BGE-large.",
        "answer": "Delta Sync Index with managed embeddings",
        "why": "Managed embeddings + Delta Sync = simplest RAG setup.",
    },
    {
        "scenario": "You use a fine-tuned sentence-transformers model to compute\n"
                    "embeddings and store them in a Delta table column.",
        "answer": "Delta Sync Index with self-managed embeddings",
        "why": "Vectors are in the table; Delta Sync handles the sync; self-managed\n"
               "because you computed the embeddings yourself.",
    },
    {
        "scenario": "You want to build a search index that updates every 6 hours\n"
                    "when a batch job adds new chunks to a Delta table.",
        "answer": "Delta Sync Index (TRIGGERED pipeline)",
        "why": "TRIGGERED pipeline + scheduled sync = periodic batch updates.",
    },
]

print("Exam Scenario Practice: Which Index Type?")
print("=" * 60)

for i, s in enumerate(scenarios, 1):
    print(f"\nScenario {i}:")
    print(f"  {s['scenario']}")
    print(f"  --> {s['answer']}")
    print(f"  Why: {s['why']}")


**Expected output:**

```
Exam Scenario Practice: Which Index Type?
============================================================

Scenario 1:
  Your company stores document chunks in a Delta table and wants
  the search index to automatically update when new documents are added.
  --> Delta Sync Index (CONTINUOUS pipeline)
  Why: Auto-sync via CDF is the defining feature of Delta Sync.

Scenario 2:
  You receive pre-computed embeddings from an external ML service
  and need to push them into a search index in real-time.
  --> Direct Vector Access Index
  Why: No Delta table source; vectors come from external system via API.
...
```

---

## Step 7: Managing Indexes -- Status, Sync, Rebuild, Delete

Once an index is created, you need to monitor and manage it. The exam may test your
knowledge of index lifecycle operations.

In [ ]:
# ---------------------------------------------------------------------------
# Index management operations
# ---------------------------------------------------------------------------
# On Databricks:
#
# # Get index reference
# index = vsc.get_index(
#     endpoint_name="genai_vs_endpoint",
#     index_name="workspace.genai_labs.document_chunks_index",
# )
#
# # Check sync status
# status = index.describe()
# print(f"State: {status['status']['detailed_state']}")
# print(f"Indexed rows: {status['status'].get('indexed_row_count', 'N/A')}")
# print(f"Ready: {status['status']['ready']}")
#
# # Trigger a sync (for TRIGGERED pipeline type)
# index.sync()
#
# # Delete an index
# vsc.delete_index(
#     endpoint_name="genai_vs_endpoint",
#     index_name="workspace.genai_labs.document_chunks_index",
# )
#
# # Delete an endpoint (removes all indexes on it)
# vsc.delete_endpoint("genai_vs_endpoint")

print("Index Management Operations")
print("=" * 50)
print()
print("1. DESCRIBE -- Check index status:")
print('   status = index.describe()')
print('   # Returns: state, indexed_row_count, ready, etc.')
print()
print("2. SYNC -- Trigger a sync (TRIGGERED pipeline only):")
print('   index.sync()')
print('   # Reads CDF and updates the index')
print()
print("3. DELETE INDEX -- Remove an index:")
print('   vsc.delete_index(')
print('       endpoint_name="genai_vs_endpoint",')
print('       index_name="workspace.genai_labs.document_chunks_index",')
print('   )')
print()
print("4. DELETE ENDPOINT -- Remove the compute resource:")
print('   vsc.delete_endpoint("genai_vs_endpoint")')
print('   # WARNING: This removes ALL indexes on the endpoint')
print()
print("5. LIST INDEXES on an endpoint:")
print('   indexes = vsc.list_indexes(endpoint_name="genai_vs_endpoint")')


**Expected output:**

```
Index Management Operations
==================================================

1. DESCRIBE -- Check index status:
   status = index.describe()

2. SYNC -- Trigger a sync (TRIGGERED pipeline only):
   index.sync()

3. DELETE INDEX -- Remove an index:
   vsc.delete_index(...)

4. DELETE ENDPOINT -- Remove the compute resource:
   vsc.delete_endpoint(...)
```

### Index States

| State | Meaning |
|---|---|
| `PROVISIONING` | Index is being created |
| `ONLINE` | Index is ready for queries |
| `OFFLINE_FAILED` | Index creation or sync failed |
| `ONLINE_TRIGGERED_SYNC` | Index is online and currently syncing new data |

In [ ]:
# ---------------------------------------------------------------------------
# UI Navigation: Viewing indexes in the Catalog
# ---------------------------------------------------------------------------
print("UI Navigation for Vector Search")
print("=" * 50)
print()
print("View your Vector Search endpoint:")
print("  **UI -->** Left nav --> Compute --> Vector Search tab")
print(f"  Find: {VS_ENDPOINT}")
print(f"  Status should be: ONLINE")
print()
print("View your index in Catalog:")
print("  **UI -->** Left nav --> Catalog --> main --> genai_labs")
print(f"  Find: document_chunks_index")
print(f"  Click to see: index type, status, source table, embedding config")
print()
print("View index details:")
print("  - Overview tab: index type, primary key, embedding config")
print("  - Status tab: sync status, indexed row count, last sync time")
print("  - Schema tab: columns available for filtering and retrieval")


**Expected output:**

```
UI Navigation for Vector Search
==================================================

View your Vector Search endpoint:
  **UI -->** Left nav --> Compute --> Vector Search tab
  ...
```

In [ ]:
# ---------------------------------------------------------------------------
# Hybrid search: combining vector similarity with keyword matching
# ---------------------------------------------------------------------------
# Databricks Vector Search supports hybrid queries that combine:
# 1. Vector similarity (semantic meaning)
# 2. Keyword matching (exact term overlap)
#
# On Databricks:
#
# hybrid_results = index.similarity_search(
#     query_text="ACID transactions in Delta Lake",
#     columns=["chunk_text", "source_doc"],
#     num_results=3,
#     query_type="HYBRID",   # Combine vector + keyword search
# )

print("Hybrid Search: Vector + Keyword")
print("=" * 50)
print()
print('hybrid_results = index.similarity_search(')
print('    query_text="ACID transactions in Delta Lake",')
print('    columns=["chunk_text", "source_doc"],')
print('    num_results=3,')
print('    query_type="HYBRID",')
print(')')
print()
print("Query types:")
print('  query_type="ANN"    --> Pure vector similarity (default)')
print('  query_type="HYBRID" --> Vector similarity + keyword matching')
print()
print("When to use HYBRID:")
print("  - Queries with specific technical terms (e.g., 'ACID', 'BGE-large')")
print("  - When exact keyword matches are important alongside semantic meaning")
print("  - When pure vector search misses results with exact term overlap")


**Expected output:**

```
Hybrid Search: Vector + Keyword
==================================================

hybrid_results = index.similarity_search(
    query_text="ACID transactions in Delta Lake",
    columns=["chunk_text", "source_doc"],
    num_results=3,
    query_type="HYBRID",
)
```

In [ ]:
# ---------------------------------------------------------------------------
# Complete retriever function for RAG
# ---------------------------------------------------------------------------
# This is the function you would use inside a RAG chain (Lab 04-03)

def retrieve_context(
    query: str,
    index_name: str = "workspace.genai_labs.document_chunks_index",
    endpoint_name: str = "genai_vs_endpoint",
    num_results: int = 3,
    columns: list = None,
    filters: dict = None,
) -> list:
    """
    Retrieve relevant chunks from a Vector Search index.
    Returns a list of dicts with chunk_text and metadata.
    """
    if columns is None:
        columns = ["chunk_text", "source_doc"]

    # Production code:
    # from databricks.vector_search.client import VectorSearchClient
    # vsc = VectorSearchClient()
    # index = vsc.get_index(endpoint_name=endpoint_name, index_name=index_name)
    #
    # search_kwargs = {
    #     "query_text": query,
    #     "columns": columns,
    #     "num_results": num_results,
    # }
    # if filters:
    #     search_kwargs["filters"] = filters
    #
    # results = index.similarity_search(**search_kwargs)
    # return results.get("result", {}).get("data_array", [])

    # Simulation
    print(f"  Searching index: {index_name}")
    print(f"  Query: '{query[:50]}...'")
    print(f"  Returning {num_results} results")
    return [
        ["Simulated chunk 1 relevant to the query.", "doc1.txt"],
        ["Simulated chunk 2 with supporting details.", "doc1.txt"],
        ["Simulated chunk 3 from another source.", "doc2.txt"],
    ][:num_results]


# Test the retriever
chunks = retrieve_context("What is Unity Catalog and how does it work?")
print()
for i, chunk in enumerate(chunks):
    print(f"  Result {i+1}: {chunk[0]} (from {chunk[1]})")


**Expected output:**

```
  Searching index: workspace.genai_labs.document_chunks_index
  Query: 'What is Unity Catalog and how does it work?...'
  Returning 3 results

  Result 1: Simulated chunk 1 relevant to the query. (from doc1.txt)
  Result 2: Simulated chunk 2 with supporting details. (from doc1.txt)
  Result 3: Simulated chunk 3 from another source. (from doc2.txt)
```

This `retrieve_context()` function is exactly what goes inside the `_retrieve()` method
of your pyfunc RAG model (Lab 04-01, Step 7).

In [ ]:
# ---------------------------------------------------------------------------
# Summary: API method cheat sheet
# ---------------------------------------------------------------------------
print("Vector Search API Cheat Sheet")
print("=" * 60)
print()
print("ENDPOINT OPERATIONS:")
print("  vsc.create_endpoint(name, endpoint_type)")
print("  vsc.list_endpoints()")
print("  vsc.delete_endpoint(name)")
print("  vsc.wait_until_endpoint_is_ready(name)")
print()
print("DELTA SYNC INDEX:")
print("  vsc.create_delta_sync_index(")
print("      endpoint_name, index_name, source_table_name,")
print("      pipeline_type, primary_key,")
print("      embedding_source_column,          # managed")
print("      embedding_model_endpoint_name,     # managed")
print("      # OR")
print("      embedding_vector_column,           # self-managed")
print("      embedding_dimension,               # self-managed")
print("  )")
print()
print("DIRECT VECTOR ACCESS INDEX:")
print("  vsc.create_direct_access_index(")
print("      endpoint_name, index_name,")
print("      primary_key, embedding_dimension,")
print("      embedding_vector_column, schema,")
print("  )")
print()
print("INDEX OPERATIONS:")
print("  index = vsc.get_index(endpoint_name, index_name)")
print("  index.similarity_search(query_text, columns, num_results)")
print("  index.describe()")
print("  index.sync()              # Delta Sync only")
print("  index.upsert(data)        # Direct Access only")
print("  index.delete(keys)        # Direct Access only")
print("  vsc.delete_index(endpoint_name, index_name)")


---

## Exam Tips

| # | Tip | Why It Matters |
|---|---|---|
| 1 | Delta Sync Index auto-syncs via CDF -- no manual updates needed | Most common exam trap: "Delta Sync requires manual sync" |
| 2 | Direct Vector Access has NO source table -- you push vectors via API | Know when to use each type |
| 3 | CDF must be enabled before creating a Delta Sync index | `delta.enableChangeDataFeed = true` |
| 4 | Managed embeddings: specify `embedding_model_endpoint_name` | Self-managed: specify `embedding_vector_column` + `embedding_dimension` |
| 5 | `query_text` for managed embeddings; `query_vector` for self-managed | The query method must match the embedding mode |
| 6 | Vector Search endpoints are separate from Model Serving endpoints | Different compute, different purpose |
| 7 | TRIGGERED pipeline syncs on demand; CONTINUOUS syncs automatically | Know the cost/latency tradeoff |
| 8 | `similarity_search()` returns a nested dict with `result.data_array` | Know the response structure for parsing results |

---

## Key Takeaways

### Index Types
- **Delta Sync Index**: auto-syncs from a Delta table via CDF. Supports managed or
  self-managed embeddings. Recommended for most RAG use cases.
- **Direct Vector Access Index**: you manage everything via API (upsert/delete).
  Self-managed embeddings only. Use for custom or external pipelines.

### Embedding Modes
- **Managed**: Databricks computes embeddings for you using an embedding model endpoint.
  Specify `embedding_source_column` and `embedding_model_endpoint_name`.
- **Self-managed**: You pre-compute vectors and store them in the table (or push via API).
  Specify `embedding_vector_column` and `embedding_dimension`.

### Querying
- `similarity_search(query_text=...)` for managed embeddings
- `similarity_search(query_vector=...)` for self-managed embeddings
- Use `filters` to narrow results by metadata
- Use `query_type="HYBRID"` for combined vector + keyword search

### Infrastructure
- Vector Search endpoints are dedicated compute, separate from Model Serving
- One endpoint can host multiple indexes
- Endpoint must be ONLINE before you can create indexes or run queries

---

## Next Lab

**Lab 04-03: RAG Deployment Pipeline** -- Now that you have a pyfunc model (Lab 04-01)
and a Vector Search index (this lab), it is time to wire them together and deploy the
complete RAG system as a serving endpoint with access control and monitoring.